In [ ]:
import fitz
print(fitz)
print(fitz.__file__)


<module 'fitz' from '/usr/local/lib/python3.12/dist-packages/fitz/__init__.py'>
/usr/local/lib/python3.12/dist-packages/fitz/__init__.py


In [ ]:
import fitz
print(fitz.__file__)
print(hasattr(fitz, "open"))


/usr/local/lib/python3.12/dist-packages/fitz/__init__.py
True


In [ ]:
!pip uninstall -y fitz
!pip uninstall -y pymupdf
!pip uninstall -y PyMuPDF


Found existing installation: PyMuPDF 1.23.8
Uninstalling PyMuPDF-1.23.8:
  Successfully uninstalled PyMuPDF-1.23.8


In [ ]:
!pip install pymupdf==1.23.8


  Using cached PyMuPDF-1.23.8-cp312-none-manylinux2014_x86_64.whl.metadata (3.4 kB)
Using cached PyMuPDF-1.23.8-cp312-none-manylinux2014_x86_64.whl (4.3 MB)


In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.1 MB/s eta 0:00:00


In [ ]:
!pip install tools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.7 MB/s eta 0:00:00


In [ ]:
!pip install fitz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.7/110.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.9/425.9 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
import fitz  # PyMuPDF
import torch
import numpy as np
from PIL import Image
import io
import base64

from transformers import (
    CLIPProcessor,
    CLIPModel,
    LlavaProcessor,
    LlavaForConditionalGeneration
)

from sklearn.metrics.pairwise import cosine_similarity
import faiss


In [ ]:
from transformers import CLIPProcessor, CLIPModel

clip_model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)
clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

clip_model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


CLIPModel(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1e-05,

In [ ]:
def embed_text(text):
    inputs = clip_processor(
        text=text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )
    with torch.no_grad():
        features = clip_model.get_text_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()


def embed_image(image):
    inputs = clip_processor(
        images=image,
        return_tensors="pt"
    )
    with torch.no_grad():
        features = clip_model.get_image_features(**inputs)
        features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()


In [ ]:
pdf_path = "/content/multimodal_sample.pdf"
doc = fitz.open(pdf_path)

documents = []
embeddings = []
image_store = {}

doc_id = 0

for page_num in range(len(doc)):
    page = doc[page_num]

    # ----- TEXT -----
    text = page.get_text().strip()
    if text:
        documents.append({
            "id": doc_id,
            "type": "text",
            "content": text,
            "page": page_num
        })
        embeddings.append(embed_text(text))
        doc_id += 1

    # ----- IMAGES -----
    for img_index, img in enumerate(page.get_images(full=True)):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]

        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

        image_store[doc_id] = image

        documents.append({
            "id": doc_id,
            "type": "image",
            "content": f"Image from page {page_num}",
            "page": page_num,
            "image_id": doc_id
        })

        embeddings.append(embed_image(image))
        doc_id += 1


In [ ]:
embedding_dim = embeddings[0].shape[0]
index = faiss.IndexFlatIP(embedding_dim)

embeddings_np = np.array(embeddings).astype("float32")
index.add(embeddings_np)


In [ ]:
def retrieve_multimodal(query, k=5):
    query_emb = embed_text(query).astype("float32").reshape(1, -1)
    scores, indices = index.search(query_emb, k)

    results = []
    for idx in indices[0]:
        results.append(documents[idx])

    return results


In [ ]:
llava_processor = LlavaProcessor.from_pretrained(
    "llava-hf/llava-1.5-7b-hf"
)

llava_model = LlavaForConditionalGeneration.from_pretrained(
    "llava-hf/llava-1.5-7b-hf",
    torch_dtype=torch.float16,
    device_map="auto"
)


preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/4.96G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

In [ ]:
def answer_with_llava(query, retrieved_docs):
    text_context = ""
    image = None

    # Collect text and ONE image
    for d in retrieved_docs:
        if d["type"] == "text":
            text_context += d["content"] + "\n"
        elif d["type"] == "image" and image is None:
            image = image_store[d["image_id"]]

    # Create prompt (with <image> token)
    prompt = f"""
<image>
You are an AI assistant answering questions using the given document.

Context:
{text_context}

Question:
{query}
"""

    # If no image was retrieved, remove <image> token
    if image is None:
        prompt = prompt.replace("<image>", "")

    # Now call LLaVA
    inputs = llava_processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(llava_model.device)

    output = llava_model.generate(
        **inputs,
        max_new_tokens=300
    )

    return llava_processor.decode(
        output[0],
        skip_special_tokens=True
    )


In [ ]:
def multimodal_rag_pipeline(query):
    retrieved_docs = retrieve_multimodal(query, k=5)

    print("Retrieved:")
    for d in retrieved_docs:
        print(f"- {d['type']} from page {d['page']}")

    answer = answer_with_llava(query, retrieved_docs)
    return answer


In [ ]:
query = "What does the chart show about revenue growth?"
response = multimodal_rag_pipeline(query)
print("\nAnswer:\n", response)


Retrieved:
- text from page 0
- image from page 0
- image from page 0
- image from page 0
- image from page 0

Answer:
 
 
You are an AI assistant answering questions using the given document.

Context:
Annual Revenue Overview
This document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart
below, revenue grew steadily with the highest growth recorded in Q3.
Q1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed
Q1 due to marketing campaigns. Q3 had exponential growth due to global expansion.


Question:
What does the chart show about revenue growth?

Answer:
The chart shows that revenue growth has been steadily increasing over the past three quarters. The highest growth was recorded in Q3, while Q1 and Q2 had moderate increases. The global expansion in Q3 contributed to the exponential growth.
